# Get CloudWatch Metrics

In [ ]:
import boto3
import pandas as pd
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo  # Python 3.9+

# -----------------------------
# CONFIG
# -----------------------------
central_tz = ZoneInfo("America/Chicago")
cloudwatch = boto3.client('cloudwatch')

# INPUT TIME FRAME IN CENTRAL TIME
start_time_ct = datetime(2025, 12, 23, 15, 0, tzinfo=central_tz)
end_time_ct = datetime(2025, 12, 23, 15, 16, tzinfo=central_tz)

# Convert Central Time → UTC (required by CloudWatch)
start_time_utc = start_time_ct.astimezone(timezone.utc)
end_time_utc = end_time_ct.astimezone(timezone.utc)

# -----------------------------
# CLOUDWATCH QUERY (UTC)
# -----------------------------
response = cloudwatch.get_metric_data(
    MetricDataQueries=[
        {
            'Id': 'lat99',
            'MetricStat': {
                'Metric': {
                    'Namespace': 'AWS/Lambda',
                    'MetricName': 'Duration',
                    'Dimensions': [
                        {'Name': 'FunctionName', 'Value': 'Lambda_CreateGet_Items'}
                    ]
                },
                'Period': 10,
                'Stat': 'Average'
            },
            'ReturnData': True,
        }
    ],
    StartTime=start_time_utc,
    EndTime=end_time_utc
)

# -----------------------------
# DATAFRAME
# -----------------------------
ts = response['MetricDataResults'][0]

df = pd.DataFrame({
    'timestamp_utc': pd.to_datetime(ts['Timestamps'], utc=True),
    'Responsetimesinms': ts['Values']
}).sort_values('timestamp_utc')

# Convert output back to Central Time
df['timestamp_ct'] = df['timestamp_utc'].dt.tz_convert(central_tz)

# -----------------------------
# PRINT RESULTS (CENTRAL TIME)
# -----------------------------
print(df[['timestamp_ct', 'Responsetimesinms']].to_string(index=False))

# Preparing Your Time-Series Data for Supervised Learning

In [ ]:
import numpy as np

series = df['Responsetimesinms'].values
window = 2

X, y = [], []
for i in range(len(series) - window - 1):
    X.append(series[i:i + window])
    y.append(series[i + window])

X = np.array(X)
y = np.array(y)

print("Number of data points:", len(series))
print("X shape:", X.shape)
print("y shape:", y.shape)

# PyTorch LSTM Time-Series Training Loop

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class ForecastLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(1, 32, batch_first=True)
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = ForecastLSTM()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

X_t = torch.tensor(X).float().unsqueeze(-1)
y_t = torch.tensor(y).float().unsqueeze(-1)

for epoch in range(30):
    optimizer.zero_grad()
    pred = model(X_t)
    loss = criterion(pred, y_t)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

# Predictions

In [ ]:
import numpy as np
import torch

model.eval()
future_steps = 15
current_window = series_scaled[-window:].reshape(-1)  # ensure 1D
predictions_scaled = []

with torch.no_grad():
    for _ in range(future_steps):
        x = torch.tensor(current_window, dtype=torch.float32).unsqueeze(0).unsqueeze(-1)  # shape (1, window, 1)
        next_val = model(x).item()
        predictions_scaled.append(next_val)
        current_window = np.append(current_window[1:], next_val)

# inverse-transform to ms
predictions = scaler.inverse_transform(np.array(predictions_scaled).reshape(-1, 1)).flatten()
for val in predictions:
    print(val)

# Plot Graph

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

# Assume df and predictions are defined as before
last_time = df['timestamp_ct'].iloc[-1]
freq = df['timestamp_ct'].diff().median()
prediction_times = [last_time + (i+1)*freq for i in range(len(predictions))]

df_pred = pd.DataFrame({
    'timestamp_ct': prediction_times,
    'Responsetimesinms': predictions
})

df_combined = pd.concat([df, df_pred], ignore_index=True)
df_combined['type'] = ['historical']*len(df) + ['predicted']*len(df_pred)
df_combined['color'] = df_combined['Responsetimesinms']

# Plot
fig = px.scatter(
    df_combined,
    x='timestamp_ct',
    y='Responsetimesinms',
    color='color',
    color_continuous_scale='Viridis',
    hover_data=['type'],
    title='CloudWatch Response Times + Predictions'
)

# Add lines connecting points
fig.add_scatter(
    x=df['timestamp_ct'],
    y=df['Responsetimesinms'],
    mode='lines',
    name='Historical',
    line=dict(color='blue')
)

fig.add_scatter(
    x=df_pred['timestamp_ct'],
    y=df_pred['Responsetimesinms'],
    mode='lines',
    name='Predicted',
    line=dict(color='red', dash='dash')
)

# Move legend below X-axis
fig.update_layout(
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.35,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)',
    ),
)

fig.show()